# Équilibre stationnaire : Vent moyen, Friction turbulente et Forçage de rappel

On cherche à vérifier l'équilibre stationnaire :

$$\frac{\partial \bar{u}}{\partial t} = 0 \quad \Longrightarrow \quad \underbrace{-\frac{\partial \overline{u'w'}}{\partial z}}_{\text{Force friction}} + \underbrace{F_{\text{rappel}}}_{-(\bar{u}-u_0)/\tau} = 0$$

Si la résultante est nulle, les deux forces s'équilibrent bien.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import xarray as xr

## 1. Chargement des données

In [ ]:
ds_ua = xr.open_dataset("3D/MESONH_RCE_small300_3D_ua.nc")
ds_va = xr.open_dataset("3D/MESONH_RCE_small300_3D_va.nc")
ds_wa = xr.open_dataset("3D/MESONH_RCE_small300_3D_wa.nc")

u = ds_ua["ua"]
v = ds_va["va"]
w = ds_wa["wa"]

# Axe temps en jours réels de simulation (75 → 100)
temps_jours = u["time"] / np.timedelta64(1, "D")
axe_z = u["altitude"].values
nom_axe_z = u.dims[1]

print(f"Période couverte : jour {temps_jours.values[0]:.1f} → jour {temps_jours.values[-1]:.1f}")
print(f"Pas de temps : {np.diff(temps_jours.values)[0]*24:.1f}h")
print(f"Nombre de pas de temps : {u.sizes['time']}")

## 2. Paramètres du forçage de rappel
**À ajuster selon ton fichier `.NAM` Meso-NH**

In [ ]:
# ============================================================
# PARAMÈTRES À AJUSTER selon ta configuration RCE
# ============================================================
u_cible  = 0.0        # Vent cible du rappel (m/s) — souvent 0 en RCE
v_cible  = 0.0
tau_rappel = 86400.0  # Temps de rappel en secondes (ex: 1 jour = 86400s)
# ============================================================

## 3. Calcul des profils temporels de vent moyen et de flux de Reynolds

In [ ]:
# Vent moyen horizontal (moyenne sur x, y) → profil (time, altitude)
u_moy = u.mean(dim=["x", "y"])   # shape: (time, altitude)
v_moy = v.mean(dim=["x", "y"])

# Décomposition de Reynolds
u_prime = u - u_moy
v_prime = v - v_moy
w_prime = w - w.mean(dim=["x", "y"])

# Flux de Reynolds → profil (time, altitude)
flux_u = (u_prime * w_prime).mean(dim=["x", "y"])
flux_v = (v_prime * w_prime).mean(dim=["x", "y"])

# Tendances friction = -d(u'w')/dz → profil (time, altitude)
tendance_friction_u = -flux_u.differentiate(nom_axe_z)
tendance_friction_v = -flux_v.differentiate(nom_axe_z)

# Forçage de rappel = -(u_moy - u_cible) / tau → profil (time, altitude)
rappel_u = -(u_moy - u_cible) / tau_rappel
rappel_v = -(v_moy - v_cible) / tau_rappel

# Résidu = friction + rappel (doit être ≈ 0 à l'état stationnaire)
residu_u = tendance_friction_u + rappel_u
residu_v = tendance_friction_v + rappel_v

# Moyennes sur la période stationnaire (toute la fenêtre 3D = déjà à l'équilibre)
u_moy_stat     = u_moy.mean(dim="time")
v_moy_stat     = v_moy.mean(dim="time")
friction_u_stat = tendance_friction_u.mean(dim="time")
friction_v_stat = tendance_friction_v.mean(dim="time")
rappel_u_stat  = rappel_u.mean(dim="time")
rappel_v_stat  = rappel_v.mean(dim="time")
residu_u_stat  = residu_u.mean(dim="time")
residu_v_stat  = residu_v.mean(dim="time")

print("Calculs terminés.")
print(f"Vent moyen max |u| : {float(np.abs(u_moy_stat).max()):.4f} m/s")
print(f"Vent moyen max |v| : {float(np.abs(v_moy_stat).max()):.4f} m/s")

## 4. Profils stationnaires moyens — Vent, Friction, Rappel, Résidu

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 8), sharey=True)

kw = dict(linewidth=2)

# --- Panneau 1 : Vent moyen ---
ax = axes[0]
ax.plot(u_moy_stat, axe_z, color="royalblue", label=r"$\bar{u}$", **kw)
ax.plot(v_moy_stat, axe_z, color="forestgreen", label=r"$\bar{v}$", **kw)
ax.plot(np.sqrt(u_moy_stat**2 + v_moy_stat**2), axe_z,
        color="black", linestyle="--", label="Module", **kw)
ax.axvline(0, color="grey", alpha=0.5)
ax.set_title("Vent moyen\nà l'état stationnaire", fontweight="bold")
ax.set_xlabel("Vitesse (m/s)")
ax.set_ylabel("Altitude (m)")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 15000)

# --- Panneau 2 : Force de friction ---
ax = axes[1]
ax.plot(friction_u_stat, axe_z, color="crimson",
        label=r"$-\partial\overline{u'w'}/\partial z$", **kw)
ax.plot(friction_v_stat, axe_z, color="darkorange",
        label=r"$-\partial\overline{v'w'}/\partial z$", **kw)
ax.axvline(0, color="grey", alpha=0.5)
ax.set_title("Force de friction\n(divergence du flux de Reynolds)", fontweight="bold")
ax.set_xlabel("Accélération ($m/s^2$)")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# --- Panneau 3 : Force de rappel ---
ax = axes[2]
ax.plot(rappel_u_stat, axe_z, color="royalblue",
        label=r"$-(\bar{u}-u_0)/\tau$", **kw)
ax.plot(rappel_v_stat, axe_z, color="forestgreen",
        label=r"$-(\bar{v}-v_0)/\tau$", **kw)
ax.axvline(0, color="grey", alpha=0.5)
ax.set_title(f"Force de rappel\n($\\tau$ = {tau_rappel/3600:.0f}h, $u_0$={u_cible} m/s)",
             fontweight="bold")
ax.set_xlabel("Accélération ($m/s^2$)")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# --- Panneau 4 : Résidu (équilibre) ---
ax = axes[3]
ax.plot(residu_u_stat, axe_z, color="black",
        label=r"Résidu $u$ (friction + rappel)", **kw)
ax.plot(residu_v_stat, axe_z, color="purple",
        label=r"Résidu $v$", linestyle="--", **kw)
ax.axvline(0, color="grey", alpha=0.5)
ax.set_title("Résidu\n(doit être ≈ 0 si équilibré)", fontweight="bold")
ax.set_xlabel("Accélération ($m/s^2$)")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.suptitle(
    "Équilibre stationnaire : Friction turbulente vs Forçage de rappel",
    fontsize=14, fontweight="bold", y=1.01
)
plt.tight_layout()
plt.savefig("equilibre_stationnaire.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Hovmöller : évolution temporelle des forces

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12), sharey=True)

donnees = [
    (u_moy,             "Vent moyen $\\bar{u}$ (m/s)",                   "RdBu_r"),
    (tendance_friction_u, "Friction $-\\partial\\overline{u'w'}/\\partial z$ ($m/s^2$)", "RdBu_r"),
    (rappel_u,           "Rappel $-(\\bar{u}-u_0)/\\tau$ ($m/s^2$)",      "RdBu_r"),
    (v_moy,             "Vent moyen $\\bar{v}$ (m/s)",                   "PuOr_r"),
    (tendance_friction_v, "Friction $-\\partial\\overline{v'w'}/\\partial z$ ($m/s^2$)", "PuOr_r"),
    (rappel_v,           "Rappel $-(\\bar{v}-v_0)/\\tau$ ($m/s^2$)",      "PuOr_r"),
]

for ax, (data, titre, cmap) in zip(axes.flat, donnees):
    vals = data.values.T   # (altitude, time)
    vmax = np.nanpercentile(np.abs(vals), 99)
    if vmax == 0: vmax = 1e-10
    norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

    cf = ax.contourf(temps_jours.values, axe_z, vals,
                     levels=40, cmap=cmap, norm=norm)
    cb = fig.colorbar(cf, ax=ax, pad=0.02)
    ax.set_title(titre, fontsize=10, fontweight="bold")
    ax.set_xlabel("Temps (jours)", fontsize=9)
    ax.set_ylabel("Altitude (m)", fontsize=9)
    ax.set_ylim(0, 15000)
    ax.grid(True, linestyle=":", alpha=0.3)

plt.suptitle(
    "Évolution temporelle : Vent moyen, Friction turbulente et Rappel (composantes U et V)",
    fontsize=13, fontweight="bold", y=1.01
)
plt.tight_layout()
plt.savefig("hovmoller_equilibre.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Séries temporelles à des altitudes clés — convergence vers l'équilibre

In [ ]:
altitudes_cibles = [200, 1000, 3000, 8000]  # mètres

fig, axes = plt.subplots(len(altitudes_cibles), 1,
                         figsize=(14, 4 * len(altitudes_cibles)), sharex=True)

for ax, alt in zip(axes, altitudes_cibles):
    fric  = tendance_friction_u.sel(altitude=alt, method="nearest")
    rap   = rappel_u.sel(altitude=alt, method="nearest")
    res   = residu_u.sel(altitude=alt, method="nearest")
    u_loc = u_moy.sel(altitude=alt, method="nearest")
    alt_exacte = float(fric.altitude)

    ax.plot(temps_jours.values, fric.values,
            color="crimson",    lw=1.8, label=r"Friction $-\partial\overline{u'w'}/\partial z$")
    ax.plot(temps_jours.values, rap.values,
            color="royalblue",  lw=1.8, label=r"Rappel $-(\bar{u}-u_0)/\tau$")
    ax.plot(temps_jours.values, res.values,
            color="black",      lw=1.2, linestyle=":", alpha=0.8, label="Résidu")

    # Axe secondaire : vent moyen
    ax2 = ax.twinx()
    ax2.plot(temps_jours.values, u_loc.values,
             color="grey", lw=1.2, linestyle="--", alpha=0.6, label=r"$\bar{u}$ (m/s)")
    ax2.set_ylabel(r"$\bar{u}$ (m/s)", color="grey", fontsize=9)
    ax2.tick_params(axis="y", labelcolor="grey")

    ax.axhline(0, color="grey", alpha=0.4, lw=0.8)
    ax.set_title(f"z ≈ {alt_exacte:.0f} m", fontweight="bold", fontsize=11)
    ax.set_ylabel("Accélération ($m/s^2$)", fontsize=9)
    ax.grid(True, linestyle="--", alpha=0.3)

    # Légendes combinées
    lines1, labs1 = ax.get_legend_handles_labels()
    lines2, labs2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labs1 + labs2, fontsize=8, loc="upper right", ncol=2)

axes[-1].set_xlabel("Temps (jours de simulation)", fontsize=11)
axes[-1].set_xlim(temps_jours.values.min(), temps_jours.values.max())

plt.suptitle(
    r"Convergence vers l'équilibre : Friction vs Rappel pour $\bar{u}$",
    fontsize=13, fontweight="bold"
)
plt.tight_layout()
plt.savefig("convergence_equilibre.png", dpi=150, bbox_inches="tight")
plt.show()